In [ ]:
import subprocess

print("Verifying environment...")

timm_version = subprocess.check_output(['pip', 'show', 'timm'], text=True).split('\n')[1].split(': ')[1]
print(f"Current timm version: {timm_version}")

version_parts = timm_version.split('.')
major = int(version_parts[0])
minor = int(version_parts[1]) if len(version_parts) > 1 else 0

if major < 1 or (major == 1 and minor < 20):
    print("Upgrading timm to >= 1.0.20 for DINOv3 support...")
    subprocess.check_call(['pip', 'install', '--upgrade', 'timm'])
    print("timm upgraded successfully")
else:
    print(f"timm version {timm_version} is compatible")

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Enable GPU in Runtime → Change runtime type")

In [ ]:
import sys
import os

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive")
    BASE_PATH = '/content/drive/MyDrive'
else:
    BASE_PATH = '.'
    print("Running locally")

print(f"Base path: {BASE_PATH}")

## Setup for Google Colab

**Before running:**

1. Download DINOv3 weights from Google Drive: https://drive.google.com/file/d/1gLJJ_sq8R8d9Jcr3to695kTlVBv0fZtI/view?usp=sharing

2. Place in your Drive at: `My Drive/dinov3/vitl16-sat493m/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth`

3. Have LandDiscover-50K dataset available in Drive at: `My Drive/data/datasets/LandDiscover_50K/`
   - Should contain: `TR_Image/` folder with 51,846 satellite images

4. Run this notebook with GPU enabled (Runtime → Change runtime type → GPU)

# Self-Supervised DINOv3 Fine-Tuning

Fine-tune DINOv3 (ViT-L/16) on LandDiscover-50K using self-supervised learning.

**Approach:** DINO-style teacher-student framework + spatial patch contrastive learning
- **DINO Loss:** Momentum teacher with centering & sharpening (prevents collapse)
- **Spatial Contrastive Loss:** InfoNCE on 8-connected patch neighbors (learns spatial structure)
- Teacher-student consistency via exponential moving average (EMA)
- Works with small batch sizes (1-2)

**Evaluation:** Train GSNet for 5K iterations with self-supervised backbones, measure mIoU vs. supervised baseline

## Setup: Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import DataLoader
from torchvision import transforms as T
from PIL import Image
import timm
import copy
from tqdm import tqdm
import cv2

print("All libraries imported successfully")
print(f"  numpy: {np.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  timm: {timm.__version__}")


In [ ]:
ROOT_DATA = os.path.join(BASE_PATH, "data/datasets/LandDiscover_50K")
IMG_DIR = os.path.join(ROOT_DATA, "TR_Image")
GT_DIR = os.path.join(ROOT_DATA, "GT_ID")
PRETRAINED_WEIGHTS = os.path.join(BASE_PATH, "dinov3/vitl16-sat493m/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth")

NUM_CLASSES = 40
IMG_SIZE = 384
BATCH_SIZE = 2
EPOCHS = 15
NUM_UNFROZEN_BLOCKS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IGNORE_INDEX = 255

MOMENTUM = 0.99
TEMPERATURE = 0.07
SPATIAL_WEIGHT = 0.5
MOMENTUM_WEIGHT = 0.5

print(f"Paths configured:")
print(f"  Data root: {ROOT_DATA}")
print(f"  Images: {IMG_DIR}")
print(f"  Weights: {PRETRAINED_WEIGHTS}")
print(f"\nHyperparameters:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Device: {DEVICE}")

## Configuration

In [ ]:
import cv2

SYNTHETIC_DATA = True
NUM_SYNTHETIC_IMAGES = 100

if os.path.isdir(IMG_DIR) and len(os.listdir(IMG_DIR)) > 0:
    SYNTHETIC_DATA = False
    print(f"Real dataset found: {len(os.listdir(IMG_DIR))} images")
    print(f"Using real LandDiscover-50K data")
else:
    print(f"Real dataset NOT found at {IMG_DIR}")
    print(f"Generating {NUM_SYNTHETIC_IMAGES} synthetic satellite images for testing...")
    
    os.makedirs(IMG_DIR, exist_ok=True)
    
    for i in range(NUM_SYNTHETIC_IMAGES):
        img = np.random.randint(0, 256, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        
        num_objects = np.random.randint(3, 8)
        for _ in range(num_objects):
            x = np.random.randint(0, IMG_SIZE - 50)
            y = np.random.randint(0, IMG_SIZE - 50)
            size = np.random.randint(20, 80)
            color = tuple(np.random.randint(50, 200, 3).tolist())
            
            if np.random.rand() > 0.5:
                cv2.rectangle(img, (x, y), (x+size, y+size), color, -1)
            else:
                cv2.circle(img, (x + size//2, y + size//2), size//2, color, -1)
        
        img_path = os.path.join(IMG_DIR, f"synthetic_{i:05d}.png")
        cv2.imwrite(img_path, img)
    
    print(f"Generated {NUM_SYNTHETIC_IMAGES} synthetic images at {IMG_DIR}")
    print("\nIMPORTANT for teammate:")
    print("- Replace IMG_DIR path above with actual LandDiscover-50K path on cluster")
    print("- Delete synthetic images if needed")
    print("- Synthetic images will be auto-detected and real data will be used instead")

print(f"\nData status:")
print(f"  Using synthetic: {SYNTHETIC_DATA}")
print(f"  Total images available: {len(os.listdir(IMG_DIR))}")

## Data Preparation: Real or Synthetic

In [ ]:
print("\nVerifying paths after data generation...")

if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(f"Image directory not found: {IMG_DIR}")

print(f"Paths verified successfully")
print(f"  Images: {IMG_DIR} ({len(os.listdir(IMG_DIR))} files)")

if not os.path.isfile(PRETRAINED_WEIGHTS):
    print(f"\nWARNING: Pretrained weights not found at {PRETRAINED_WEIGHTS}")
    print(f"  Download from: https://drive.google.com/file/d/1gLJJ_sq8R8d9Jcr3to695kTlVBv0fZtI/view")
    print(f"  Place at: {PRETRAINED_WEIGHTS}")
    print(f"  Code will continue but will fail at model loading (expected for testing)")
    WEIGHTS_AVAILABLE = False
else:
    weights_size = os.path.getsize(PRETRAINED_WEIGHTS) / 1e9
    print(f"  Weights: {PRETRAINED_WEIGHTS} ({weights_size:.2f} GB)")
    WEIGHTS_AVAILABLE = True

## Dataset: Images Only (No Labels)

In [ ]:
class LandDiscoverUnsupervisedDataset(Dataset):
    def __init__(self, img_root, transform=None):
        self.img_root = img_root
        self.transform = transform
        
        self.files = sorted([
            f for f in os.listdir(img_root)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ])
        
        assert len(self.files) > 0, "No images found in TR_Image"
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        name = self.files[idx]
        img = Image.open(os.path.join(self.img_root, name)).convert("RGB")
        
        if self.transform:
            img1 = self.transform[0](img)
            img2 = self.transform[1](img)
            return img1, img2
        
        return img, img

print("Dataset class defined")

## Augmentation: Weak and Strong

In [ ]:
weak_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

strong_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=90),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

transforms = [weak_transform, strong_transform]
print("Augmentation pipelines defined")
print("  Weak: flip + rotate")
print("  Strong: flip + rotate + color jitter + affine + blur")

## DataLoader

In [ ]:
dataset = LandDiscoverUnsupervisedDataset(IMG_DIR, transform=transforms)
print(f"Dataset size: {len(dataset)}")

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoader created")
print(f"  Total batches per epoch: {len(loader)}")

## Load DINOv3 Backbone

In [ ]:
backbone = timm.create_model(
    "vit_large_patch16_dinov3.sat493m",
    pretrained=False,
    num_classes=0,
    dynamic_img_size=True,
    img_size=IMG_SIZE
).to(DEVICE)

print(f"Created backbone: ViT-L/16 DINOv3")
print(f"  Embedding dimension: {backbone.embed_dim}")
print(f"  Number of blocks: {len(backbone.blocks)}")
print(f"  Patch size: {backbone.patch_embed.patch_size}")

## Load Pretrained SAT-493M Weights

In [ ]:
if not WEIGHTS_AVAILABLE:
    print(f"Skipping weight loading (weights not available)")
    print(f"Backbone will use random initialization for testing")
else:
    print(f"Loading pretrained weights from: {PRETRAINED_WEIGHTS}")
    ckpt = torch.load(PRETRAINED_WEIGHTS, map_location="cpu", weights_only=False)

    if "teacher" in ckpt:
        backbone.load_state_dict(ckpt["teacher"], strict=False)
        print("Loaded from 'teacher' key")
    elif "teacher_state_dict" in ckpt:
        backbone.load_state_dict(ckpt["teacher_state_dict"], strict=False)
        print("Loaded from 'teacher_state_dict' key")
    elif "model" in ckpt:
        backbone.load_state_dict(ckpt["model"], strict=False)
        print("Loaded from 'model' key")
    else:
        backbone.load_state_dict(ckpt, strict=False)
        print("Loaded raw state dict")

    print("Pretrained weights loaded successfully")

## Projection Head (MLP for Self-Supervised Loss)

In [ ]:
class ProjectionHead(nn.Module):
    """3-layer MLP projection head for self-supervised learning.
    
    Maps backbone features (1024-d for ViT-L) into lower-dimensional space (256-d)
    where loss is computed. Prevents representation collapse.
    """
    def __init__(self, input_dim=1024, hidden_dim=2048, output_dim=256):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        """
        Args:
            x: [B, N, D] token features from backbone
        Returns:
            [B, N, output_dim] projected features (L2-normalized)
        """
        projected = self.mlp(x)
        return F.normalize(projected, dim=-1, p=2)

student_projection_head = ProjectionHead(input_dim=backbone.embed_dim, output_dim=256).to(DEVICE)

print(f"Projection head created:")
print(f"  Input dim: {backbone.embed_dim}")
print(f"  Hidden dim: 2048")
print(f"  Output dim: 256")
print(f"  Total params: {sum(p.numel() for p in student_projection_head.parameters()):,}")

## Freeze/Unfreeze Blocks

In [ ]:
for p in backbone.parameters():
    p.requires_grad = False

print(f"Unfreezing last {NUM_UNFROZEN_BLOCKS} blocks...")
for block in backbone.blocks[-NUM_UNFROZEN_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True

total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
trainable_pct = 100 * trainable_params / total_params

print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({trainable_pct:.1f}%)")

## Teacher-Student Models (Momentum Encoder)

## Self-Supervised Loss Functions

In [ ]:
class DINOLoss(nn.Module):
    """DINO Loss with centering and sharpening to prevent collapse.
    
    Teacher output is centered (subtract running mean) and sharpened (low temperature).
    Student must match the teacher's sharp distribution via cross-entropy.
    """
    def __init__(self, out_dim=256, teacher_temp=0.04, student_temp=0.1, center_momentum=0.9):
        super().__init__()
        self.teacher_temp = teacher_temp
        self.student_temp = student_temp
        self.center_momentum = center_momentum
        
        # Running center for teacher outputs (prevents collapse)
        self.register_buffer("center", torch.zeros(1, out_dim))
    
    def forward(self, student_output, teacher_output):
        """
        Args:
            student_output: [B, N, D] projected student features
            teacher_output: [B, N, D] projected teacher features
        Returns:
            cross-entropy loss between distributions
        """
        # Mean pool over patches to get global representation
        student_out = student_output.mean(dim=1)  # [B, D]
        teacher_out = teacher_output.mean(dim=1)  # [B, D]
        
        # Teacher: center then sharpen
        teacher_out = teacher_out - self.center
        teacher_out = F.softmax(teacher_out / self.teacher_temp, dim=-1)
        teacher_out = teacher_out.detach()  # Stop gradient
        
        # Student: sharpen (no centering)
        student_out = F.log_softmax(student_out / self.student_temp, dim=-1)
        
        # Cross-entropy loss
        loss = -torch.sum(teacher_out * student_out, dim=-1).mean()
        
        return loss
    
    def update_center(self, teacher_output):
        """Update running center after each batch"""
        batch_center = teacher_output.mean(dim=1).mean(dim=0, keepdim=True)  # [1, D]
        self.center = self.center * self.center_momentum + batch_center * (1 - self.center_momentum)


class SpatialContrastiveLoss(nn.Module):
    """Vectorized spatial patch contrastive loss with InfoNCE objective.
    
    Neighboring patches (8-connected grid) should be similar (positives).
    Non-neighboring patches should be dissimilar (negatives).
    """
    def __init__(self, temperature=0.07, grid_size=24):
        super().__init__()
        self.temperature = temperature
        self.grid_size = grid_size
        
        # Precompute neighbor mask [N, N] where N = grid_size^2
        self.register_buffer("neighbor_mask", self._build_neighbor_mask(grid_size))
    
    def _build_neighbor_mask(self, grid_size):
        """Build binary mask: neighbor_mask[i, j] = 1 if patches i and j are neighbors"""
        N = grid_size * grid_size
        mask = torch.zeros(N, N, dtype=torch.bool)
        
        for i in range(N):
            h_i, w_i = i // grid_size, i % grid_size
            
            # 8-connected neighbors
            for dh in [-1, 0, 1]:
                for dw in [-1, 0, 1]:
                    if dh == 0 and dw == 0:
                        continue
                    
                    h_j, w_j = h_i + dh, w_i + dw
                    if 0 <= h_j < grid_size and 0 <= w_j < grid_size:
                        j = h_j * grid_size + w_j
                        mask[i, j] = True
        
        return mask
    
    def forward(self, feat_tokens):
        """
        Args:
            feat_tokens: [B, N, D] projected patch features
        Returns:
            InfoNCE loss
        """
        B, N, D = feat_tokens.shape
        
        # Compute full similarity matrix [B, N, N]
        feat_norm = F.normalize(feat_tokens, dim=-1, p=2)
        sim_matrix = torch.matmul(feat_norm, feat_norm.transpose(1, 2)) / self.temperature  # [B, N, N]
        
        # For each anchor, compute InfoNCE loss
        # L_i = -log(sum_positives(exp(sim)) / sum_all(exp(sim)))
        
        # Mask for positives (neighbors)
        pos_mask = self.neighbor_mask.unsqueeze(0).expand(B, -1, -1)  # [B, N, N]
        
        # Compute log-sum-exp for numerator (positives only)
        sim_pos = sim_matrix.masked_fill(~pos_mask, -1e9)
        log_prob_pos = torch.logsumexp(sim_pos, dim=-1)  # [B, N]
        
        # Compute log-sum-exp for denominator (all except self)
        # Mask out diagonal (self-similarity)
        diag_mask = torch.eye(N, dtype=torch.bool, device=feat_tokens.device).unsqueeze(0).expand(B, -1, -1)
        sim_all = sim_matrix.masked_fill(diag_mask, -1e9)
        log_prob_all = torch.logsumexp(sim_all, dim=-1)  # [B, N]
        
        # InfoNCE loss: -log(exp(pos) / exp(all)) = log(all) - log(pos)
        loss = (log_prob_all - log_prob_pos).mean()
        
        return loss


# Instantiate loss functions
dino_loss = DINOLoss(out_dim=256, teacher_temp=0.04, student_temp=0.1).to(DEVICE)
spatial_loss = SpatialContrastiveLoss(temperature=TEMPERATURE, grid_size=IMG_SIZE // 16).to(DEVICE)

print("Loss functions defined:")
print("  1. DINO Loss: teacher-student consistency with centering & sharpening")
print("     - Teacher temp: 0.04 (sharp)")
print("     - Student temp: 0.1 (softer)")
print("     - Center momentum: 0.9")
print("  2. Spatial Contrastive Loss: InfoNCE with 8-connected neighbors")
print(f"     - Grid size: {IMG_SIZE // 16}x{IMG_SIZE // 16}")
print(f"     - Temperature: {TEMPERATURE}")
print(f"     - Neighbor mask shape: {spatial_loss.neighbor_mask.shape}")

## Optimizer

In [ ]:
backbone_lr = 5e-5
projection_lr = 5e-4

optimizer = AdamW(
    [
        {"params": student_model.blocks[-NUM_UNFROZEN_BLOCKS:].parameters(), "lr": backbone_lr},
        {"params": student_projection_head.parameters(), "lr": projection_lr},
    ],
    weight_decay=1e-4
)

print(f"Optimizer configured:")
print(f"  Type: AdamW")
print(f"  Backbone LR: {backbone_lr}")
print(f"  Projection head LR: {projection_lr}")
print(f"  Weight decay: 1e-4")

## Helper Functions

In [ ]:
def get_num_special_tokens(feat_tokens, img_size=IMG_SIZE, patch_size=16):
    """Calculate number of special tokens (CLS + registers) in DINOv3 output"""
    B, N, C = feat_tokens.shape
    grid_size = img_size // patch_size
    expected_patches = grid_size * grid_size
    num_special_tokens = N - expected_patches
    return num_special_tokens

def extract_patch_features(feat_tokens, img_size=IMG_SIZE, patch_size=16):
    """Extract spatial patch tokens (remove CLS + register tokens)"""
    B, N, C = feat_tokens.shape
    
    grid_size = img_size // patch_size
    expected_patches = grid_size * grid_size
    num_special_tokens = N - expected_patches
    
    # Remove special tokens (CLS + registers)
    spatial_tokens = feat_tokens[:, num_special_tokens:, :]
    
    # Reshape to spatial grid [B, C, H, W]
    feat_map = spatial_tokens.transpose(1, 2).reshape(
        B, C, grid_size, grid_size
    )
    
    return feat_map

def extract_patch_tokens(feat_tokens, img_size=IMG_SIZE, patch_size=16):
    """Extract spatial patch tokens as sequence [B, N_patches, C]"""
    num_special_tokens = get_num_special_tokens(feat_tokens, img_size, patch_size)
    return feat_tokens[:, num_special_tokens:, :]

def save_checkpoint(epoch, backbone, projection_head, optimizer, output_dir="./checkpoints"):
    os.makedirs(output_dir, exist_ok=True)
    
    path = os.path.join(output_dir, f"self_supervised_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "backbone": backbone.state_dict(),
        "projection_head": projection_head.state_dict(),
        "optimizer": optimizer.state_dict(),
    }, path)
    
    backbone_only_path = os.path.join(output_dir, f"backbone_only_epoch_{epoch}.pth")
    torch.save(backbone.state_dict(), backbone_only_path)
    
    print(f"Checkpoint saved:")
    print(f"  Full: {path}")
    print(f"  Backbone: {backbone_only_path}")

print("Helper functions defined")
print("  - get_num_special_tokens: compute CLS + register count")
print("  - extract_patch_features: spatial grid [B, C, H, W]")
print("  - extract_patch_tokens: token sequence [B, N, C]")
print("  - save_checkpoint: saves backbone + projection head")

## Training Loop

In [ ]:
print(f"Starting self-supervised fine-tuning...\n")

for epoch in range(EPOCHS):
    student_model.train()
    student_projection_head.train()
    teacher_model.eval()
    teacher_projection_head.eval()
    
    total_loss = 0.0
    dino_loss_sum = 0.0
    spatial_loss_sum = 0.0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx, (img_weak, img_strong) in enumerate(pbar):
        img_weak = img_weak.to(DEVICE)
        img_strong = img_strong.to(DEVICE)
        
        # Teacher forward (no gradients)
        with torch.no_grad():
            teacher_feat_tokens = teacher_model.forward_features(img_weak)
            teacher_proj_output = teacher_projection_head(teacher_feat_tokens)
        
        # Student forward
        student_feat_tokens = student_model.forward_features(img_strong)
        student_proj_output = student_projection_head(student_feat_tokens)
        
        # Extract patch tokens (remove CLS + registers)
        student_patch_tokens = extract_patch_tokens(student_proj_output)
        teacher_patch_tokens = extract_patch_tokens(teacher_proj_output)
        
        # Loss 1: DINO loss (teacher-student consistency)
        dino_loss_val = dino_loss(student_proj_output, teacher_proj_output)
        
        # Loss 2: Spatial contrastive loss (neighboring patches should be similar)
        spatial_loss_val = spatial_loss(student_patch_tokens)
        
        # Combined loss
        total_loss_val = MOMENTUM_WEIGHT * dino_loss_val + SPATIAL_WEIGHT * spatial_loss_val
        
        # Backward and optimize
        optimizer.zero_grad()
        total_loss_val.backward()
        optimizer.step()
        
        # Update teacher via EMA
        update_teacher(student_model, teacher_model, student_projection_head, teacher_projection_head, tau=MOMENTUM)
        
        # Update DINO center (for collapse prevention)
        dino_loss.update_center(teacher_proj_output)
        
        # Track losses
        total_loss += total_loss_val.item()
        dino_loss_sum += dino_loss_val.item()
        spatial_loss_sum += spatial_loss_val.item()
        
        pbar.set_postfix({
            "loss": f"{total_loss_val.item():.4f}",
            "dino": f"{dino_loss_val.item():.4f}",
            "spa": f"{spatial_loss_val.item():.4f}"
        })
    
    avg_loss = total_loss / len(loader)
    avg_dino_loss = dino_loss_sum / len(loader)
    avg_spatial_loss = spatial_loss_sum / len(loader)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Total Loss: {avg_loss:.4f}")
    print(f"  DINO Loss: {avg_dino_loss:.4f}")
    print(f"  Spatial Loss: {avg_spatial_loss:.4f}")
    
    # Save checkpoints periodically
    if (epoch + 1) % 5 == 0 or epoch == 0:
        save_checkpoint(epoch+1, student_model, student_projection_head, optimizer)

print("\nSelf-supervised fine-tuning completed!")

## Save Final Models

In [ ]:
os.makedirs("./checkpoints", exist_ok=True)

final_path = "./checkpoints/self_supervised_final.pth"
torch.save({
    "epoch": EPOCHS,
    "backbone": student_model.state_dict(),
    "projection_head": student_projection_head.state_dict(),
    "optimizer": optimizer.state_dict(),
}, final_path)

final_backbone_path = "./checkpoints/self_supervised_backbone_final.pth"
torch.save(student_model.state_dict(), final_backbone_path)

print("Final checkpoints saved:")
print(f"  Full checkpoint: {final_path}")
print(f"  Backbone only: {final_backbone_path}")
print(f"\nTo use with GSNet:")
print(f"  Set RSIB_CKPT to: {os.path.abspath(final_backbone_path)}")
print(f"\nNote: Projection head is NOT used in GSNet (only for self-supervised training)")

## Summary

In [ ]:
print("="*60)
print("Self-Supervised Fine-Tuning Summary")
print("="*60)
print(f"\nModel: DINOv3 ViT-L/16")
print(f"Dataset: LandDiscover-50K (images only)")
print(f"Total images: {len(dataset)}")
print(f"\nTraining Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Unfrozen blocks: {NUM_UNFROZEN_BLOCKS}")
print(f"  Trainable params: {trainable_params:,} ({trainable_pct:.1f}%)")
print(f"\nArchitecture:")
print(f"  Backbone: DINOv3 ViT-L/16 ({backbone.embed_dim}-dim)")
print(f"  Projection head: 3-layer MLP (1024 → 2048 → 2048 → 256)")
print(f"  Teacher-student: EMA momentum = {MOMENTUM}")
print(f"\nLoss Functions:")
print(f"  1. DINO Loss (weight={MOMENTUM_WEIGHT}):")
print(f"     - Centering with momentum = 0.9")
print(f"     - Teacher temp = 0.04 (sharp)")
print(f"     - Student temp = 0.1 (softer)")
print(f"  2. Spatial Contrastive Loss (weight={SPATIAL_WEIGHT}):")
print(f"     - InfoNCE with 8-connected neighbors")
print(f"     - Temperature = {TEMPERATURE}")
print(f"     - Grid size: {IMG_SIZE // 16}x{IMG_SIZE // 16}")
print(f"\nHyperparameters:")
print(f"  Backbone LR: {backbone_lr}")
print(f"  Projection head LR: {projection_lr}")
print(f"  Weight decay: 1e-4")
print(f"\nCheckpoints saved in: ./checkpoints/")
print(f"\nNext step:")
print(f"  1. Train GSNet with the fine-tuned backbone")
print(f"  2. Evaluate mIoU on test datasets (FloodNet, FLAIR, FAST, Potsdam)")
print(f"  3. Compare with supervised fine-tuning baseline")
print("="*60)